# DLP Insider Threat Detection — Full Pipeline

**Before running:**
1. Set Runtime → Change runtime type → **T4 GPU**
2. Get your Kaggle API key: kaggle.com → Account → API → *Create New Token* → downloads `kaggle.json`
3. Run all cells top to bottom

**Pipeline steps**
| Step | What happens |
|------|--------------|
| 1 | Clone repo from GitHub |
| 2 | Install dependencies |
| 3 | Upload Kaggle credentials |
| 4 | Download CERT dataset from Kaggle |
| 5 | Configure paths |
| 6 | Clean all raw data |
| 7 | Train Isolation Forest (baseline) |
| 8 | Visualize Isolation Forest results |
| 9 | Train LSTM Autoencoder (GPU) |
| 10 | Visualize LSTM results |
| 11 | (Optional) Save outputs to Google Drive |

## Step 1 — Clone repo from GitHub

In [ ]:
import os

REPO_URL = 'https://github.com/Taha-coder-star/DLP-PROJECt.git'
REPO_DIR = '/content/dlp-project'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print('Working directory:', os.getcwd())

## Step 2 — Install dependencies

In [ ]:
!pip install -r requirements.txt kaggle -q
print('Done.')

## Step 3 — Upload Kaggle credentials
Go to **kaggle.com → Account → API → Create New Token** to download `kaggle.json`, then run this cell and upload it.

In [ ]:
from google.colab import files

print('Upload your kaggle.json file:')
uploaded = files.upload()  # select kaggle.json from your computer

import os, shutil
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.move('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('Kaggle credentials configured.')

## Step 4 — Download CERT dataset from Kaggle
Find the dataset slug on Kaggle: open the dataset page → look at the URL → `kaggle.com/datasets/USERNAME/DATASET-NAME`.
Set `KAGGLE_DATASET` below to `USERNAME/DATASET-NAME`.

In [ ]:
# ===================== UPDATED v4 =====================
KAGGLE_DATASET = 'mrajaxnp/cert-insider-threat-detection-research'
ARCHIVE_DIR    = '/content/dlp-data/archive'

import os, zipfile
from pathlib import Path
os.makedirs(ARCHIVE_DIR, exist_ok=True)

REQUIRED_FILES = [
    'email.csv', 'psychometric.csv', 'logon.csv',
    'device.csv', 'file.csv', 'decoy_file.csv', 'users.csv',
]

archive = Path(ARCHIVE_DIR)

for fname in REQUIRED_FILES:
    dest    = archive / fname
    zip_path = archive / (fname + '.zip')

    # unzip if zip is present but CSV is missing
    if zip_path.exists() and not dest.exists():
        print(f'  Unzipping {zip_path.name} ...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(ARCHIVE_DIR)
        zip_path.unlink()

    if dest.exists():
        print(f'  {fname} already present.')
        continue

    # download (will save as fname.zip), then unzip
    print(f'  Downloading {fname} ...')
    !kaggle datasets download -d {KAGGLE_DATASET} -f {fname} -p {ARCHIVE_DIR} --force
    if zip_path.exists():
        print(f'  Unzipping {zip_path.name} ...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(ARCHIVE_DIR)
        zip_path.unlink()

# verify
found = sorted(archive.glob('*.csv'))
print(f'\nFound {len(found)} CSV files:')
for f in found:
    print(f'  {f.name:40s} {f.stat().st_size / 1_048_576:8.1f} MB')

missing = set(REQUIRED_FILES) - {f.name for f in found}
if missing:
    print('WARNING: missing files:', missing)
else:
    print('All required files present.')

## Step 5 — Configure paths

In [ ]:
import os
from pathlib import Path

# All data lives in Colab local storage for this session
os.environ['DLP_ROOT'] = '/content/dlp-data'

root = Path(os.environ['DLP_ROOT'])
for d in ['archive', 'cleaned', 'models', 'plots']:
    (root / d).mkdir(parents=True, exist_ok=True)

print('DLP_ROOT :', os.environ['DLP_ROOT'])
print('GPU      :', end=' ')
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT available — go to Runtime > Change runtime type > T4 GPU')

## Step 6 — Clean all raw data
> Takes ~10–20 min (processes ~10M rows across all files).

In [ ]:
!python scripts/clean_cert_email_data.py

## Step 7 — Train Isolation Forest (baseline)

In [ ]:
!python colab/train_isolation_forest_cert.py

## Step 8 — Visualize Isolation Forest results

In [ ]:
!python colab/visualize_isolation_forest_cert.py

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import os

for img in sorted((Path(os.environ['DLP_ROOT']) / 'plots').glob('iforest_*.png')):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(mpimg.imread(img))
    ax.axis('off')
    ax.set_title(img.stem)
    plt.tight_layout()
    plt.show()

## Step 9 — Train LSTM Autoencoder (GPU)
> Trains one model per user (1,000 models). Takes ~30–60 min on T4 GPU.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU found! Go to Runtime > Change runtime type > T4 GPU and re-run from Step 5.'
print('GPU ready:', torch.cuda.get_device_name(0))

In [ ]:
!python colab/train_lstm_autoencoder_cert.py

## Step 10 — Visualize LSTM results

In [ ]:
!python colab/visualize_lstm_autoencoder_cert.py

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import os

for img in sorted((Path(os.environ['DLP_ROOT']) / 'plots').glob('lstm_*.png')):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(mpimg.imread(img))
    ax.axis('off')
    ax.set_title(img.stem)
    plt.tight_layout()
    plt.show()

## Step 11 — Launch Monitoring Dashboard
Run the interactive Streamlit app to explore model results, compare IForest vs LSTM, and investigate users.

In [ ]:
# Install localtunnel to expose the Streamlit app publicly from Colab
!pip install streamlit pyngrok -q

import threading, time
from pyngrok import ngrok

# Kill any previous tunnel
ngrok.kill()

# Start Streamlit in background
def run_streamlit():
    import subprocess
    subprocess.run([
        'streamlit', 'run', 'app/monitoring_app.py',
        '--server.port', '8501',
        '--server.headless', 'true',
        '--server.enableCORS', 'false',
    ])

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(5)  # wait for streamlit to start

# Open public tunnel
tunnel = ngrok.connect(8501)
print('=' * 60)
print('Dashboard URL:', tunnel.public_url)
print('=' * 60)
print('Open the URL above in your browser.')
print('The app has 6 tabs:')
print('  1. Pipeline Overview — data splits, model configs')
print('  2. Isolation Forest  — scores, severity, top users')
print('  3. LSTM Autoencoder  — scores, user timeline')
print('  4. Model Comparison  — IForest vs LSTM correlation')
print('  5. User Investigation — per-user deep dive')
print('  6. Live Scoring       — score any row in real-time')